# DEMO: Dashboard Interactivity in the UI

Following your preferences, this notebook is now focused on the learner workflow in the dashboard UI.

This is the third notebook in the series. It builds on the dashboard and visualizations created in the earlier demos and teaches the learner how to add interactivity to that draft dashboard.

By the end of this demo, the learner should know how to:
* add static widget-level filters to a single visualization
* add interactive page-level and global filters
* understand the difference between dashboard filters and dataset-level filtering
* create SQL datasets with parameters and build visualizations from them
* create and use dashboard variables
* test cross-filtering and drill-through in the UI
* create custom calculations and compare them with local metric views and Unity Catalog metric views

Keep this demo scoped to dashboard authoring in the UI. It is not about rebuilding datasets from scratch, publishing, or unrelated advanced platform topics.

---

## Step 1: Add Filters in the UI and Understand Filter Scope

### A. Add a static widget-level filter

Widget-level filters affect only one visualization and are set by the dashboard author.

1. Click an existing visualization widget on the canvas.
2. In the configuration panel, confirm the dataset selection.
3. Click **Show filters**.
4. Under **Filter fields**, click the **plus** icon and choose a field.
5. Set the filter value you want for that one widget.
6. Save the visualization.

Use this when you want one chart to stay fixed on a subset of data, such as only one region or one product category.

### B. Add a page-level filter widget

Page-level filters are interactive controls that viewers can change.

1. Click **Add a filter (field/parameter)** and place the filter widget on the current page.
2. In the configuration panel, choose the filter type you want.
3. Add one or more dataset fields that should respond to the filter.
4. Give the filter a clear title.
5. Save the filter widget.

Use this when you want one page to have its own controls without affecting the rest of the dashboard.

### C. Add a global filter

Global filters apply across pages for visualizations that share one or more datasets.

1. Click the **filter** icon in the upper-left corner of the dashboard canvas.
2. Click **+** to create a new global filter.
3. Configure it the same way you would configure a filter widget.
4. Choose the dataset field or fields that should respond.
5. Save the filter.

Use this for dashboard-wide controls such as a shared date range or region selector.

### Dashboard filters versus dataset-level filtering

Dashboard filters work in the dashboard UI. Dataset-level filtering is built into the dataset itself.

| Filter layer | Where it is defined | What it does | When to use it |
| --- | --- | --- | --- |
| Widget-level filter | Visualization configuration | Filters one widget only | Keep one chart fixed to a subset |
| Page-level filter | Filter widget on one page | Lets viewers filter widgets on that page | Page-specific exploration |
| Global filter | Global filter panel | Lets viewers filter multiple pages | Shared dashboard controls |
| SQL dataset filter | SQL query in the Data tab | Restricts rows returned by the dataset itself | Logic that should always apply |
| Local metric view logic | Local metric view definition in the Data tab | Defines dashboard-scoped measures, dimensions, and joins | Dashboard-only semantic logic |
| Unity Catalog metric view | Unity Catalog metric view definition | Provides governed shared measures and dimensions | Reusable semantic logic across dashboards |

Important distinctions for the learner:
* a SQL `WHERE` clause is a dataset-level filter because it changes the dataset result before the visualization is built
* a direct Unity Catalog metric view dataset cannot be edited to exclude columns or apply dataset-query filters in the editor; if you need to reshape or filter it at the dataset level, create a SQL dataset that references the metric view and use `MEASURE()` for measures
* local metric views keep semantic logic inside the dashboard, while Unity Catalog metric views make that logic reusable across dashboards and other tools

---

In [0]:
%sql
-- Step 2 example: create a SQL dataset with a dashboard parameter.
-- Paste this into a new SQL dataset in the Data tab.

SELECT
  product_category,
  SUM(net_revenue) AS total_revenue,
  COUNT(order_id) AS order_count,
  try_divide(SUM(profit), SUM(net_revenue)) AS profit_margin
FROM gold_sales
WHERE region = :selected_region
GROUP BY product_category
ORDER BY total_revenue DESC

## Step 2: Create a Parameterized Dataset and Build a Visualization from It

Use the SQL in the previous cell to create a parameter-driven dataset.

1. Open the **Data** tab.
2. Click **Add SQL dataset**.
3. Paste in the SQL from the previous cell.
4. Click **Run**.
5. Notice that Databricks detects `:selected_region` and creates a parameter control above the editor.
6. Configure the parameter so it is easy for a learner to understand, such as a dropdown or combobox labeled **Selected Region**.
7. Save the dataset with a clear name such as `revenue_by_category_parameterized`.

Now connect the parameter to the dashboard canvas.

1. Return to the **Canvas** tab.
2. Add a **filter widget**.
3. In the filter configuration, choose the parameter `selected_region`.
4. If you want a query-based dropdown, connect the parameter to a field such as `gold_sales.region` so the widget can show existing values.
5. Add a bar chart from the new dataset.
6. Set the category field to `product_category`.
7. Set the value field to `total_revenue`.
8. Title the visualization **Revenue by Category for Selected Region**.

### Filters versus parameters

| Topic | Filter | Parameter |
| --- | --- | --- |
| Operates at | Dashboard layer | Dataset query layer |
| Typical use | Narrow an existing dataset | Pass a value into SQL at runtime |
| Best for | Simple interactive slicing | Dynamic query logic, date ranges, and conditions |
| Viewer experience | Usually field-based and direct | Often bound to a filter widget |
| Trade-off | Simpler but less flexible | More flexible but requires SQL design |

Teach the learner this rule of thumb:
* use a filter when the field already exists in the dataset and you only need to narrow results
* use a parameter when the dataset query itself must change based on user input

---

## Step 3: Create and Use a Dashboard Variable

Dashboard variables are different from filters and parameters.

A dashboard variable lets the viewer switch which field a visualization displays. It does not filter rows, and it does not substitute values into SQL.

### Example workflow

1. Create a new visualization or open an existing one that uses a dataset such as `vw_region_performance` or `mv_sales_metrics`.
2. In the visualization editor, choose one encoding that you want the viewer to switch, such as the Y-axis.
3. Use the variable control in that encoding to create a variable.
4. Add two field options of the same general type, such as `total_revenue` and `total_profit`, or two compatible dimensions.
5. Bind the variable to a control widget.
6. Save the visualization.
7. Change the variable selection and confirm that the visualization updates to show a different field.

If you add the same variable to more than one visualization, one control can switch all of them together.

### Variables versus filters versus parameters

| Tool | What changes | Best use |
| --- | --- | --- |
| Filter | Which rows are shown | Narrow the visible data |
| Parameter | The dataset SQL at runtime | Change query logic or query inputs |
| Variable | Which field a chart encodes | Let viewers swap measures or dimensions |

Important learner takeaway:
* use a variable when the user should be able to switch the field shown on a chart without changing the dataset query
* use a parameter when the dataset query itself needs a runtime input
* use a filter when the user is simply narrowing data that is already available in the dataset

---

## Step 4: Test Cross-Filtering and Set Up Drill-Through

### A. Cross-filtering on the same page

Cross-filtering lets a viewer click a data point in one visualization and filter other visualizations that use the same dataset.

1. On the **Interactivity Lab** page, add two visualizations from the same dataset.
2. A simple choice is a bar chart and a table that both use `vw_region_performance`, or two visuals that both use `gold_sales`.
3. Save both widgets on the same page.
4. Enter preview mode or interact with the draft.
5. Click a bar, pie slice, point, or other supported mark in one visualization.
6. Confirm that the other visualization updates to the selected segment.

Use this to show learners how they can drill into a subset without adding another filter widget.

Important rule:
* cross-filtering automatically applies to supported visualizations that use the same dataset

### B. Drill-down versus drill-through

Use these terms clearly in the demo:
* **Drill-down on the same page**: click a chart selection to narrow what other widgets show through cross-filtering
* **Drill-through to another page**: move to a target page with filters populated from the selected segment

### C. Set up drill-through

1. Add a second page and name it **Details**.
2. Place one or more visualizations on that page that use the same dataset as the source visualization.
3. Return to the source page.
4. Right-click a chart segment in the source visualization.
5. Choose **Drill to** and select the target page.
6. Confirm that the target page opens with the selected context applied.

If the target page also includes filters based on the same dataset, those filters can populate automatically from the drill-through selection.

---

In [0]:
%sql
-- Optional second parameter example: reference a Unity Catalog metric view in SQL.
-- Use this when you want dataset-level filtering or reshaping on top of a metric view.
-- When querying a metric view in SQL, measures must use MEASURE().

SELECT
  `Product Category`,
  MEASURE(`Total Revenue`) AS total_revenue,
  MEASURE(`Order Count`) AS order_count,
  MEASURE(`Avg Order Value`) AS avg_order_value
FROM mv_sales_metrics
WHERE `Region` = :selected_region
GROUP BY ALL
ORDER BY total_revenue DESC

## Step 5: Create Custom Calculations and Build a Visualization from Them

Custom calculations are created in the dashboard UI and stay with the dataset in that dashboard.

### Create a calculated measure

1. Open an existing dataset in the **Data** tab, such as `gold_sales`.
2. Click **Add custom calculation**.
3. Create a calculated measure named `profit_margin`.
4. Use an expression such as `try_divide(SUM(profit), SUM(net_revenue))`.
5. Save the calculation.

### Create a calculated dimension

1. Click **Add custom calculation** again.
2. Create a calculated dimension named `order_value_band`.
3. Use a row-level expression such as `CASE WHEN net_revenue >= 1000 THEN 'High value order' ELSE 'Standard order' END`.
4. Save the calculation.

### Build a visualization from the custom fields

1. Return to the **Canvas** tab.
2. Add a visualization that uses the dataset where you created the custom calculations.
3. Use `order_value_band` as the category field.
4. Use `profit_margin` as the value field.
5. Title the chart **Profit Margin by Order Value Band**.
6. Save the visualization.

This shows the learner that custom calculations can create new analytical fields without changing the dataset SQL.

---

## Step 6: Compare Custom Calculations, Local Metric Views, and Unity Catalog Metric Views

The learner should understand that these three tools solve different problems.

### Custom calculations
* created inside a dashboard dataset
* useful for quick calculated measures and calculated dimensions
* scoped to that dataset and dashboard
* best for dashboard-specific analysis

### Local metric views
* created in the **Data** tab by clicking **Add data** and choosing the option to create a metric view
* built with a visual interface for dimensions, measures, and joins
* scoped to the dashboard
* useful when you want dashboard-scoped semantic logic, not just one or two ad-hoc calculations
* can also extend an existing Unity Catalog metric view
* currently a newer feature, so call out that it is dashboard-scoped rather than globally shared

### Unity Catalog metric views
* created and governed in Unity Catalog
* reusable across dashboards and other tools
* best for shared business definitions
* if selected directly as a dataset, the dashboard uses the metric view as defined
* if referenced in a SQL dataset, you can reshape or filter it, but measures must use `MEASURE()`

### Quick decision guide

| Need | Best fit |
| --- | --- |
| One dashboard-only measure or dimension | Custom calculation |
| Dashboard-scoped semantic layer with measures, dimensions, and joins | Local metric view |
| Shared governed semantic logic across dashboards | Unity Catalog metric view |

Teach the learner this progression:
1. Start with a custom calculation for quick experimentation.
2. Move to a local metric view when the dashboard needs richer semantic logic.
3. Promote important shared logic to a Unity Catalog metric view when it should become reusable across dashboards.

---

---

## Summary

By the end of this notebook, the learner should have practiced all of the following in the dashboard UI:
* adding a static widget-level filter
* adding a page-level filter widget
* adding a global filter
* distinguishing dashboard filters from dataset-level filtering
* creating a SQL dataset with a parameter
* binding a parameter to a filter widget and building a visualization from it
* creating and using a dashboard variable
* testing cross-filtering on the same page
* setting up drill-through to another page
* creating a custom measure and a custom dimension
* understanding how custom calculations differ from local metric views and Unity Catalog metric views

That keeps this third demo aligned with the series: the learner first created datasets, then built visualizations, and now adds interactivity in the UI.
